In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.html) \|\|
[Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| **Build Model** \|\|
[Autograd](autogradqs_tutorial.html) \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

Build the Neural Network
========================

Neural networks comprise of layers/modules that perform operations on
data. The [torch.nn](https://pytorch.org/docs/stable/nn.html) namespace
provides all the building blocks you need to build your own neural
network. Every module in PyTorch subclasses the
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
A neural network is a module itself that consists of other modules
(layers). This nested structure allows for building and managing complex
architectures easily.

In the following sections, we\'ll build a neural network to classify
images in the FashionMNIST dataset.


In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

Get Device for Training
=======================

We want to be able to train our model on an
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If the current accelerator is
available, we will use it. Otherwise, we use the CPU.


In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


Define the Class
================

We define our neural network by subclassing `nn.Module`, and initialize
the neural network layers in `__init__`. Every `nn.Module` subclass
implements the operations on input data in the `forward` method.


In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

We create an instance of `NeuralNetwork`, and move it to the `device`,
and print its structure.


In [4]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


To use the model, we pass it the input data. This executes the model\'s
`forward`, along with some [background
operations](https://github.com/pytorch/pytorch/blob/270111b7b611d174967ed204776985cefca9c144/torch/nn/modules/module.py#L866).
Do not call `model.forward()` directly!

Calling the model on the input returns a 2-dimensional tensor with dim=0
corresponding to each output of 10 raw predicted values for each class,
and dim=1 corresponding to the individual values of each output. We get
the prediction probabilities by passing it through an instance of the
`nn.Softmax` module.


In [61]:
X = torch.rand(1, 1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([8])


------------------------------------------------------------------------


Model Layers
============

Let\'s break down the layers in the FashionMNIST model. To illustrate
it, we will take a sample minibatch of 3 images of size 28x28 and see
what happens to it as we pass it through the network.


In [18]:
input_image = torch.rand(3,28,28)
print(input_image.size())
print(input_image)

torch.Size([3, 28, 28])
tensor([[[0.0167, 0.7717, 0.9102,  ..., 0.1254, 0.6202, 0.7937],
         [0.3865, 0.7923, 0.5304,  ..., 0.5759, 0.5867, 0.2449],
         [0.3649, 0.8914, 0.7714,  ..., 0.1525, 0.5939, 0.6497],
         ...,
         [0.8112, 0.3861, 0.1468,  ..., 0.2462, 0.7741, 0.4491],
         [0.1836, 0.1633, 0.4682,  ..., 0.9088, 0.3961, 0.6241],
         [0.9653, 0.8403, 0.3250,  ..., 0.2651, 0.1014, 0.5719]],

        [[0.0045, 0.9568, 0.3541,  ..., 0.6601, 0.9900, 0.6654],
         [0.9025, 0.1960, 0.2629,  ..., 0.1591, 0.0320, 0.2794],
         [0.5003, 0.5632, 0.1004,  ..., 0.2966, 0.3625, 0.6047],
         ...,
         [0.0653, 0.5500, 0.3340,  ..., 0.0664, 0.0610, 0.2264],
         [0.8276, 0.9104, 0.3100,  ..., 0.5755, 0.4802, 0.3569],
         [0.9123, 0.7932, 0.9913,  ..., 0.1958, 0.6329, 0.1239]],

        [[0.0644, 0.2135, 0.9445,  ..., 0.6992, 0.6025, 0.6406],
         [0.1100, 0.8449, 0.2182,  ..., 0.2806, 0.8478, 0.7164],
         [0.4552, 0.5673, 0.0754, 

nn.Flatten
==========

We initialize the
[nn.Flatten](https://docs.pytorch.org/docs/stable/generated/torch.nn.modules.flatten.Flatten.html)
layer to convert each 2D 28x28 image into a contiguous array of 784
pixel values ( the minibatch dimension (at dim=0) is maintained).


In [19]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())
print(flat_image)

torch.Size([3, 784])
tensor([[0.0167, 0.7717, 0.9102,  ..., 0.2651, 0.1014, 0.5719],
        [0.0045, 0.9568, 0.3541,  ..., 0.1958, 0.6329, 0.1239],
        [0.0644, 0.2135, 0.9445,  ..., 0.3945, 0.4646, 0.9749]])


nn.Linear
=========

The [linear
layer](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
is a module that applies a linear transformation on the input using its
stored weights and biases.


In [21]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())
print(hidden1)

torch.Size([3, 20])
tensor([[ 0.2878,  0.0971,  0.2931, -0.1519, -0.2503,  0.4937, -0.4298,  0.4298,
         -0.4904, -0.2179,  0.3045, -0.2799,  0.4028, -0.1094,  0.2219, -0.4802,
         -0.1135, -0.2481, -0.1519,  0.2385],
        [ 0.2091,  0.0307,  0.0703, -0.4022, -0.5486,  0.1904, -0.2857,  0.1557,
          0.0179, -0.2876,  0.6043, -0.2300,  0.2343,  0.1357,  0.1111, -0.1995,
         -0.0684, -0.1423, -0.3720,  0.1633],
        [ 0.2333,  0.0206, -0.0151, -0.1894, -0.5923, -0.1709, -0.3686,  0.4629,
         -0.2536, -0.5474,  0.7477, -0.2660,  0.2738,  0.1583, -0.0535, -0.2938,
         -0.1652,  0.0811, -0.4246, -0.1530]], grad_fn=<AddmmBackward0>)


nn.ReLU
=======

Non-linear activations are what create the complex mappings between the
model\'s inputs and outputs. They are applied after linear
transformations to introduce *nonlinearity*, helping neural networks
learn a wide variety of phenomena.

In this model, we use
[nn.ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
between our linear layers, but there\'s other activations to introduce
non-linearity in your model.


In [22]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.2878,  0.0971,  0.2931, -0.1519, -0.2503,  0.4937, -0.4298,  0.4298,
         -0.4904, -0.2179,  0.3045, -0.2799,  0.4028, -0.1094,  0.2219, -0.4802,
         -0.1135, -0.2481, -0.1519,  0.2385],
        [ 0.2091,  0.0307,  0.0703, -0.4022, -0.5486,  0.1904, -0.2857,  0.1557,
          0.0179, -0.2876,  0.6043, -0.2300,  0.2343,  0.1357,  0.1111, -0.1995,
         -0.0684, -0.1423, -0.3720,  0.1633],
        [ 0.2333,  0.0206, -0.0151, -0.1894, -0.5923, -0.1709, -0.3686,  0.4629,
         -0.2536, -0.5474,  0.7477, -0.2660,  0.2738,  0.1583, -0.0535, -0.2938,
         -0.1652,  0.0811, -0.4246, -0.1530]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.2878, 0.0971, 0.2931, 0.0000, 0.0000, 0.4937, 0.0000, 0.4298, 0.0000,
         0.0000, 0.3045, 0.0000, 0.4028, 0.0000, 0.2219, 0.0000, 0.0000, 0.0000,
         0.0000, 0.2385],
        [0.2091, 0.0307, 0.0703, 0.0000, 0.0000, 0.1904, 0.0000, 0.1557, 0.0179,
         0.0000, 0.6043, 0.0000, 0.2343, 0.1357, 0.11

nn.Sequential
=============

[nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html)
is an ordered container of modules. The data is passed through all the
modules in the same order as defined. You can use sequential containers
to put together a quick network like `seq_modules`.


In [23]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

nn.Softmax
==========

The last linear layer of the neural network returns [logits]{.title-ref}
- raw values in \[-infty, infty\] - which are passed to the
[nn.Softmax](https://pytorch.org/docs/stable/generated/torch.nn.Softmax.html)
module. The logits are scaled to values \[0, 1\] representing the
model\'s predicted probabilities for each class. `dim` parameter
indicates the dimension along which the values must sum to 1.


In [24]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

Model Parameters
================

Many layers inside a neural network are *parameterized*, i.e. have
associated weights and biases that are optimized during training.
Subclassing `nn.Module` automatically tracks all fields defined inside
your model object, and makes all parameters accessible using your
model\'s `parameters()` or `named_parameters()` methods.

In this example, we iterate over each parameter, and print its size and
a preview of its values.


In [25]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[-0.0284, -0.0167,  0.0100,  ..., -0.0106,  0.0306,  0.0033],
        [-0.0289,  0.0055, -0.0012,  ..., -0.0146,  0.0328,  0.0032]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([ 0.0186, -0.0173], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0316, -0.0239,  0.0429,  ...,  0.0009, -0.0084, -0.0072],
        [ 0.0388,  0.0083,  0.0014,  ..., -0.0362,  0.0128,  0.0112]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | 

------------------------------------------------------------------------


Further Reading
===============

-   [torch.nn API](https://pytorch.org/docs/stable/nn.html)
